# nb_03_gold: conformed star schema
**Task 3.** Build five Gold tables from Silver. Nothing here exists in any source file: tenure, the calendar, duration and in-window days are all derived.

In [ ]:
from pyspark.sql import functions as F

# Reporting window: 1 June to 30 June 2026 inclusive = 30 days. END is the exclusive edge (1 July).
WINDOW_START = "2026-06-01"
WINDOW_END   = "2026-07-01"
TENURE_AS_OF = "2026-06-30"       # fixed date so the tenure numbers do not change when you re-run later

def save(df, name):
    (df.write.mode("overwrite").format("delta")
       .option("overwriteSchema", "true").saveAsTable(name))
    print(f"{name:20s} rows = {spark.table(name).count()}")

## gold_dim_customer: tenure_years and tenure_band (neither is in any source)

In [ ]:
sc = spark.table("silver_customers")
tenure = F.floor(F.datediff(F.lit(TENURE_AS_OF).cast("date"), F.col("registered_on")) / 365.25)

gold_dim_customer = (sc
    .withColumn("tenure_years", tenure.cast("int"))
    .withColumn("tenure_band",
        F.when(F.col("tenure_years") < 1, "0-1 yrs")
         .when(F.col("tenure_years") < 2, "1-2 yrs")
         .when(F.col("tenure_years") < 3, "2-3 yrs")
         .otherwise("3+ yrs"))
    .select("customer_id", "customer_name", "customer_type", "city",
            "registered_on", "kyc_verified_on", "tenure_years", "tenure_band"))
save(gold_dim_customer, "gold_dim_customer")

## gold_dim_depot: already clean, promote as-is

In [ ]:
save(spark.table("silver_depots").select("depot_code", "depot_name", "zone", "fleet_size"), "gold_dim_depot")

## gold_dim_date: a 30-day calendar generated here (no source sends one)

In [ ]:
gold_dim_date = (spark.sql(f"SELECT explode(sequence(DATE'{WINDOW_START}', DATE'2026-06-30', INTERVAL 1 DAY)) AS full_date")
    .select(
        F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
        "full_date",
        F.dayofmonth("full_date").alias("day_of_month"),
        F.date_format("full_date", "EEEE").alias("day_name"),
        F.weekofyear("full_date").alias("week_of_year"),
        F.month("full_date").alias("month_num"),
        F.date_format("full_date", "MMMM").alias("month_name"),
        F.when(F.dayofweek("full_date").isin(1, 7), 1).otherwise(0).alias("is_weekend")))
save(gold_dim_date, "gold_dim_date")

## gold_fact_rental: keep EVERY rental, returned or not

In [ ]:
sr = spark.table("silver_rentals")

checkout_d = F.to_date("checkout_ts")
checkin_d  = F.to_date("checkin_ts")             # NULL while the machine is still out
w_start = F.lit(WINDOW_START).cast("date")
w_end   = F.lit(WINDOW_END).cast("date")

gold_fact_rental = (sr
    .withColumn("asset_type", F.split("asset_id", "-").getItem(0))
    .withColumn("checkout_date", checkout_d)
    # NULL (not 0) while the machine is out: 0 would claim it came back instantly
    .withColumn("rental_duration_days", F.datediff(checkin_d, checkout_d))
    .withColumn("is_returned", F.when(F.col("checkin_ts").isNotNull(), 1).otherwise(0))
    # days rented INSIDE the window: clip both ends. A still-out machine counts up to the window edge, no further.
    .withColumn("asset_days_in_window",
        F.greatest(F.lit(0),
            F.datediff(F.least(F.coalesce(checkin_d, w_end), w_end),
                       F.greatest(checkout_d, w_start))))
    .select("rental_id", "customer_id", "depot_code", "asset_id", "asset_type", "rental_type",
            "checkout_ts", "checkin_ts", "checkout_date",
            "rental_duration_days", "is_returned", "asset_days_in_window"))
save(gold_fact_rental, "gold_fact_rental")

## gold_fact_billing: the parsed bills

In [ ]:
save(spark.table("silver_billing").select("bill_id", "rental_id", "bill_date", "amount_inr", "payer_type"),
     "gold_fact_billing")

## OPTIMIZE ... ZORDER and DESCRIBE HISTORY (Fabric / Delta only)

In [ ]:
spark.sql("OPTIMIZE gold_fact_rental ZORDER BY (depot_code, customer_id)")
spark.sql("OPTIMIZE gold_fact_billing ZORDER BY (rental_id)")
display(spark.sql("DESCRIBE HISTORY gold_fact_rental"))
display(spark.sql("DESCRIBE HISTORY gold_fact_billing"))

**Why no partitioning:** these tables hold under 1,000 rows; partitioning adds a folder and file per value and only pays off at hundreds of millions of rows.
**Why duration is NULL, not 0:** a machine still out has no duration yet; 0 would claim it came back instantly and would drag every average down.

## Acceptance checks (screenshot this cell's output)

In [ ]:
expected = {"gold_dim_customer": 600, "gold_dim_depot": 6, "gold_dim_date": 30,
            "gold_fact_rental": 942, "gold_fact_billing": 779}
for t, n in expected.items():
    got = spark.table(t).count()
    assert got == n, f"{t}: expected {n}, got {got}"
    print(f"OK  {t:20s} {got}")

fr = spark.table("gold_fact_rental")
still_out = fr.filter(F.col("checkin_ts").isNull())
assert still_out.count() == 175
assert still_out.filter(F.col("rental_duration_days").isNotNull()).count() == 0, "duration must be NULL while out"
assert still_out.filter(F.col("is_returned") != 0).count() == 0
assert fr.filter(F.col("checkin_ts").isNotNull() & F.col("rental_duration_days").isNull()).count() == 0
mx = fr.agg(F.max("asset_days_in_window")).first()[0]
assert mx <= 30, mx
print(f"OK  rental_duration_days is NULL for exactly {still_out.count()} still-out rentals, is_returned = 0 for them")
print(f"OK  max asset_days_in_window = {mx} (never above 30)")
display(spark.table("gold_dim_customer").groupBy("tenure_band").count().orderBy("tenure_band"))